# 08 — Published EMNLP Urdu benchmark audit

**Purpose:** Verify the exact original eight-category test set; prepare a separate task only if independent eight-category TRAINING data exists.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# 0. CLONE + ENVIRONMENT SETUP — RUN FIRST (Kaggle/Colab).
# Internet must be ON. The FIRST run may install libraries and STOP for a kernel restart.
# After restarting the kernel, rerun this cell; it will skip installation if versions match.
from pathlib import Path
import os, sys, subprocess, importlib.metadata as metadata

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
PROJECT_FOLDER = "Urdu_Qwen_Research_Notebooks"
PULL_LATEST = False  # Set True after pushing changes to GitHub; requires a clean Git checkout.

if Path('/kaggle/working').is_dir():
    WORKSPACE = Path('/kaggle/working')
elif Path('/content').is_dir():
    WORKSPACE = Path('/content')
else:
    WORKSPACE = Path.cwd()
REPO_DIR = WORKSPACE / 'nlp_research_proj'
ROOT = REPO_DIR / PROJECT_FOLDER

if (REPO_DIR / '.git').is_dir():
    if PULL_LATEST:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    if REPO_DIR.exists():
        raise RuntimeError(f'{REPO_DIR} exists but is not a Git clone. Rename/remove it after backing up outputs.')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
if not (ROOT / 'requirements.txt').is_file() or not (ROOT / 'project_utils.py').is_file():
    raise FileNotFoundError(f'Expected complete project at {ROOT}. Check the GitHub folder layout.')

# This environment was selected for a stable Transformers 4 training API.
if not ((3, 10) <= sys.version_info[:2] <= (3, 12)):
    raise RuntimeError(f'Python {sys.version.split()[0]} not in tested support range 3.10–3.12.')
REQUIRED = {
    'transformers': '4.51.3',
    'peft': '0.15.2',
    'accelerate': '1.6.0',
    'datasets': '3.5.0',
    'tokenizers': '0.21.1',
    'huggingface-hub': '0.30.2',
    'safetensors': '0.5.3',
    'scikit-learn': '1.6.1',
    'pandas': '2.2.3',
    'numpy': '1.26.4',
    'matplotlib': '3.10.1',
    'psutil': '7.0.0',
    'ipywidgets': '8.1.6',
    'nbformat': '5.10.4',
    'pytest': '8.3.5',
}

def installed_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

mismatch = {name: (installed_version(name), version)
            for name, version in REQUIRED.items()
            if installed_version(name) != version}
torchao = installed_version('torchao')
if mismatch or torchao is not None:
    print('Installing the project-locked environment; CUDA-enabled PyTorch is left untouched.')
    if mismatch:
        print('Packages to align:', mismatch)
    if torchao is not None:
        print('Removing optional incompatible torchao:', torchao)
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt'),
                    '--upgrade-strategy', 'only-if-needed'], check=True)
    raise RuntimeError('Installation finished. RESTART THE KAGGLE/COLAB KERNEL, then rerun this FIRST cell. '
                       'Do not train before restarting.')

# No package installation needed: import the shared research code only now.
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()

# Catch future API drift before downloading a 0.5B model.
import inspect
from transformers import TrainingArguments, Trainer
sig = inspect.signature(TrainingArguments).parameters
needed = ['output_dir', 'group_by_length', 'warmup_ratio', 'eval_strategy']
missing = [name for name in needed if name not in sig]
if missing:
    raise RuntimeError(f'Transformers does not match our 4.51.3 training API. Missing: {missing}')
api = check_training_api()
print('Ready. Project:', ROOT)
print('Transformers:', api['transformers'], '| PEFT:', api['peft'])
print('Trainer token argument:', api['trainer_tokenizer_key'])
print('No installation needed; proceed to the next cell.')


**Critical:** The [paper repository](https://github.com/sameearif/Generalists-vs-Specialists) gives the *test* data under `datasets/topic-classification.csv` with columns `input,label`. It has **8 labels**, unlike SIB-200's 7. Do **not** train on the paper test set or map SIB-200 labels into these eight categories. Published 0.8474 macro-F1 applies to the paper's test set, not SIB-200.

In [ ]:
# Download the public repository. Kaggle Internet or Colab Internet must be enabled.
import subprocess
RUN_CLONE=False
repo_dir=ROOT/'data/external/Generalists-vs-Specialists'
repo_dir.parent.mkdir(parents=True,exist_ok=True)
if RUN_CLONE and not repo_dir.exists():
    subprocess.check_call(['git','clone','--depth','1',cfg()['paper_repo_url'],str(repo_dir)])
paper_test=repo_dir/'datasets/topic-classification.csv'
if paper_test.exists():
    items=read_csv(paper_test)
    print('Paper test rows:',len(items),'columns:',list(items[0]))
    print('Test label counts:',__import__('collections').Counter(r['label'] for r in items))
else:
    print('Clone repo first, then re-run. Expected:',paper_test)

In [ ]:
# Audit, not training. Every label ID must map to the repository's documented label_mapping.py.
if paper_test.exists():
    test_raw=read_csv(paper_test)
    assert set(test_raw[0]) >= {'input','label'}
    assert all(str(r['label']).strip() in [str(x) for x in range(8)] for r in test_raw)
    print('SIB-200 labels:',DEFAULT_LABELS)
    print('Published paper 8 labels:',PAPER_LABELS)
    print('Same class set?',set(DEFAULT_LABELS)==set(PAPER_LABELS))
    print('Paper test is held out and MUST NOT be used for training/validation.')

### Optional 8-class reproduction

Provide a **separate** `paper8_train.csv` and `paper8_validation.csv` with `text,label` columns, using *exactly* the eight published label strings. These files must come from a licensed training source that you document. No rows from the published test CSV may appear in them. If no legitimate train/validation set can be found, stop here and report the published score as related work only.

In [ ]:
TRAIN_FILE=ROOT/'data/external/paper8_train.csv'
VALID_FILE=ROOT/'data/external/paper8_validation.csv'
RUN_PAPER8_IMPORT=False
if RUN_PAPER8_IMPORT:
    assert paper_test.exists() and TRAIN_FILE.exists() and VALID_FILE.exists(), 'Missing paper test or independent training data'
    train, val, test = read_csv(TRAIN_FILE),read_csv(VALID_FILE),read_csv(paper_test)
    assert all(set(r)>= {'text','label'} and r['label'] in PAPER_LABELS for r in train+val)
    test_rows=[]
    for i,r in enumerate(test):
        label=PAPER_LABELS[int(r['label'])]
        test_rows.append({'example_id':stable_id('paper8','ur','test',i),'index_id':'','source':'paper EMNLP 2024','lang':'ur','split':'test','text':r['input'],'label':label,'label_id':int(r['label'])})
    def make_rows(src,split):
        return [dict(example_id=stable_id('paper8','ur',split,i),index_id='',source='independent_training',
                     lang='ur',split=split,text=r['text'],label=r['label'],label_id=PAPER_LABELS.index(r['label'])) for i,r in enumerate(src)]
    allrows=make_rows(train,'train')+make_rows(val,'validation')+test_rows
    problems=validate_split(allrows,PAPER_LABELS)
    assert not problems, f'Leakage or malformed data: {problems}'
    savedir=ROOT/'data/paper8';savedir.mkdir(parents=True,exist_ok=True)
    for split in ['train','validation','test']:
        write_csv(savedir/f'ur_{split}.csv',[x for x in allrows if x['split']==split])
    write_json(savedir/'manifest.json',dict(task='paper8',labels=PAPER_LABELS,original_test_sha256=file_hash(paper_test),
               separate_training_source='DOCUMENT YOUR TRAINING DATA HERE',
               notes='Paper eight-category test set; independently sourced training & validation; exact fixed evaluation'))
    print('Paper8 task prepared. To train: set TASK=paper8 in notebooks 03/04 and only select feasible per-class sizes.')
else: print('Paper8 import disabled; audit mode only. This avoids using held-out test data as training.')